In [147]:
import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split

import torchmetrics

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode

In [148]:
seed = 42
root_path = "/home/stefan/ioai-prep/kits/neoai/terminal_animals"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(seed)

# Data

In [149]:
train_df = pd.read_csv(f"{root_path}/train.csv")
test_df = pd.read_csv(f"{root_path}/test.csv")
train_df.head()

,img,label
0,\\/\\\\?—\\ccccc/ccccco\P/c;.|o\ooPPPPPPP/PPPP...,32
1,\P/P|\\\ooo||o—||\\@/■\■■■\\■■——\\■■\||■\@\\/|...,32
2,\\\\\\/—\—/c/|\|c\cccc/\c\/|\/c\c/|cc—/—/\//\/...,34
3,—/—////??@@■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■...,24
4,\\/\/\\///\\/\/\|c\|o/o||/|\\..../c\ ...,19


In [150]:
tr, ev = train_test_split(train_df, test_size=0.15, random_state=seed, stratify=train_df["label"])
tr.shape, ev.shape

((3128, 2), (552, 2))

In [151]:
all_text = "".join(train_df["img"].tolist())
vocab = sorted(list(set(all_text)))
char_to_idx = {char: i + 1 for i, char in enumerate(vocab)}
char_to_idx["<UNK>"] = 0
vocab_size = len(char_to_idx)
vocab_size

16

In [152]:
def text_to_idx(img_str):
    lines = img_str.strip().split("\n")
    grid = np.zeros((128, 128), dtype=np.int64)
    for i, line in enumerate(lines[:128]):
        line_indices = [char_to_idx.get(c, 0) for c in line[:128]]
        grid[i, : len(line_indices)] = line_indices
    return grid

In [153]:
class ASCIIDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        grid = text_to_idx(self.df.iloc[idx]["img"])
        img_tensor = torch.from_numpy(grid).unsqueeze(0).to(torch.float32)

        if self.transform:
            img_tensor = self.transform(img_tensor)

        img_tensor = img_tensor.squeeze(0).to(torch.long)

        if self.is_test:
            return img_tensor
        return img_tensor, torch.tensor(self.df.iloc[idx]["label"], dtype=torch.long)

In [154]:
num_classes = train_df["label"].nunique()

train_transforms = transforms.Compose([
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), interpolation=InterpolationMode.NEAREST),
])

train_loader = DataLoader(
    ASCIIDataset(tr, transform=train_transforms), batch_size=64, shuffle=True
)
val_loader = DataLoader(ASCIIDataset(ev), batch_size=64, shuffle=False)

# Model

In [155]:
class ResNetASCII(nn.Module):
    def __init__(self, vocab_size, num_classes, emb_dim=64):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.stem = nn.Sequential(
            nn.Conv2d(emb_dim, 64, 3, 1, 3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1),
        )

        self.resnet = models.resnet34(weights=None)
        self.resnet.conv1 = self.resnet.bn1 = self.resnet.relu = self.resnet.maxpool = nn.Identity()
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5), 
            nn.Linear(self.resnet.fc.in_features, num_classes)
        )

    def forward(self, x):
        x = self.embedding(x).permute(0, 3, 1, 2)
        return self.resnet(self.stem(x))

In [156]:
model = ResNetASCII(vocab_size, num_classes).to(device)

# Training

In [157]:
best_f1 = 0
epochs = 300
patience = 20
no_improve = 0

f1_metric = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average="micro").to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=5e-2)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [158]:
for epoch in range(1, epochs+1):
    model.train()
    train_loss = 0
    for imgs, labels in tqdm(train_loader):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    with torch.no_grad():
        for imgs, labels in val_loader:
            pred = model(imgs.to(device)).argmax(1)
            f1_metric.update(pred, labels.to(device))

    val_f1 = f1_metric.compute()
    f1_metric.reset()
    scheduler.step()

    if val_f1 > best_f1:
        best_f1 = val_f1
        no_improve = 0
        torch.save(model.state_dict(), f"{root_path}/best_v2.pth")
        print(f"New Best F1: {val_f1:.4f}")
    else:
        no_improve += 1

    print(f"Epoch {epoch} | Loss: {train_loss/len(train_loader):.4f} | F1: {val_f1:.4f}")

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

100%|██████████| 49/49 [00:13<00:00,  3.72it/s]


New Best F1: 0.0453
Epoch 1 | Loss: 3.8241 | F1: 0.0453


100%|██████████| 49/49 [00:12<00:00,  3.89it/s]


New Best F1: 0.0507
Epoch 2 | Loss: 3.5776 | F1: 0.0507


100%|██████████| 49/49 [00:13<00:00,  3.71it/s]


...

Epoch 103 | Loss: 0.8965 | F1: 0.3859


100%|██████████| 49/49 [00:14<00:00,  3.46it/s]


Epoch 104 | Loss: 0.8428 | F1: 0.4239


100%|██████████| 49/49 [00:13<00:00,  3.57it/s]


Epoch 105 | Loss: 0.7720 | F1: 0.4257
Early stopping at epoch 105


# Submission

In [159]:
model.load_state_dict(torch.load(f"{root_path}/best_v2.pth"))
test_loader = DataLoader(ASCIIDataset(test_df, is_test=True), batch_size=48)
model.eval()
final_preds = []
with torch.no_grad():
    for imgs in tqdm(test_loader):
        out = model(imgs.to(device))
        final_preds.extend(out.argmax(1).cpu().numpy())

100%|██████████| 77/77 [00:07<00:00,  9.77it/s]


In [160]:
test_df["label"] = final_preds
test_df.to_csv(f"{root_path}/submission.csv", columns=["id", "label"], index=None)